# Python for Poets 2026 - Workbook 3 - Finding distinctive words

## Setup

In [ ]:
#@title Importing code libraries
import pandas as pd
import nltk
from collections import Counter
from tqdm.auto import tqdm
import plotnine as p9
import warnings
from scipy.stats import chi2_contingency
from functools import lru_cache
cache = lru_cache(maxsize=None)
warnings.filterwarnings('ignore')


nltk.download('punkt_tab')  # necessary for word tokenization
nltk.download('stopwords')  # necessary for word tokenization
stopwords = set(nltk.corpus.stopwords.words('english'))
stopwords.add('applause')

In [ ]:
#@title Download corpus (U.S. State of the Union speeches)
!wget -nc -q "https://www.dropbox.com/s/6gyueael6smbxyg/sotu_metadata.zip?dl=1" -O sotu_metadata.zip
!wget -nc -q "https://www.dropbox.com/s/w73qio0thhfzdpx/sotu_txt.zip?dl=1" -O sotu_txt.zip
!unzip -n -q sotu_metadata.zip
!unzip -n -q sotu_txt.zip
!mv metadata.csv sotu_metadata.csv
!mv txt sotu_txt

In [ ]:
#@title Open corpus metadata spreadsheet via pandas
import pandas as pd
df_corpus = pd.read_csv('sotu_metadata.csv')
df_corpus = df_corpus.sort_values('year')
df_corpus

In [ ]:
#@title Get party affiliations
df_parties = pd.read_csv('https://docs.google.com/spreadsheets/d/e/2PACX-1vSdeAmHi47hihQ1JxK3SmoBBVH7sOCK7qcBioALss9T2EeOFV57Gi0tI7qmX0_RS_AAdZtkKmi0cxyI/pub?gid=0&single=true&output=csv')
df_parties = df_parties.fillna('')
df_parties

In [ ]:
#@title Merge party affiliations with corpus dataset
df_merged = df_corpus.merge(df_parties, left_on='year', right_on='Years (after inauguration)', how='left')
df_merged

## Exploring corpus

In [ ]:
#@title Show number of speeches per party
df_merged['Party'].value_counts()

In [ ]:
#@title TODO: Show number of speeches per president
df_merged['REPLACE_ME'].value_counts()

In [ ]:
#@title Open text files and add text as column to df
texts = []
for id in df_merged['id']:
    with open(f'sotu_txt/{id}.txt') as f:
        txt = f.read()
        texts.append(txt)
df_merged['txt'] = texts
df_merged

In [ ]:
#@title Tokenize and count words
counts_D = Counter()
counts_R = Counter()

# Loop over each row
for i,row in tqdm(df_merged.iterrows(), total=len(df_merged)):

    # Lowercase the txt
    txt_lowercase = row['txt'].lower()

    # Tokenize the text
    tokens = nltk.word_tokenize(txt_lowercase)
    tokens = [t for t in tokens if t not in stopwords and t.isalpha()]

    # Update the right counter
    if row['Party'] == 'Democrat':
        counts_D.update(tokens)
    elif row['Party'] == 'Republican':
        counts_R.update(tokens)

print('10 most common words for Democrats:')
print(counts_D.most_common(10))
print()
print('10 most common words for Republicans:')
print(counts_R.most_common(10))

In [ ]:
#@title Compute relative frequencies

# get total counts
total_D = sum(counts_D.values())
total_R = sum(counts_R.values())

# get relative count (measured as frequency per million or `fpm`)
fpm_D = {word: count/total_D * 1_000_000 for word,count in counts_D.items()}
fpm_R = {word: count/total_R * 1_000_000 for word,count in counts_R.items()}

# look up a single word
word = 'government'

fpm_D[word], fpm_R[word]

In [ ]:
#@title Try a different word

word = 'REPLACE_ME'
fpm_D[word], fpm_R[word]

In [ ]:
#@title Compute frequency difference
shared_words = set(fpm_D.keys()) & set(fpm_R.keys())
freq_diff = {word: fpm_R[word] - fpm_D[word] for word in shared_words}
freq_diff['immigration'], freq_diff['working']

In [ ]:
freq_diff_sorted = sorted(freq_diff.items(), key=lambda x: x[1])

print('Top 10 most differentially frequent words for Democrats:')
print(freq_diff_sorted[:10])
print()
print('Top 10 most differentially frequent words for Republicans:')
print(freq_diff_sorted[-10:])

In [ ]:
#@title TODO: Get another words' difference in frequency

freq_diff['REPLACE_ME']

## Plotting

In [ ]:
#@title (Functions to get counts from a text quickly)

@cache
def get_count_data_for_text(txt):
    txt_lowercase = txt.lower()
    tokens = nltk.word_tokenize(txt_lowercase)
    tokens = [t for t in tokens if t not in stopwords and t.isalpha()]
    counts = Counter(tokens)
    return counts

@cache
def get_word_freq_historical_data(word):
    ld = []

    for i,row in tqdm(df_merged.iterrows(), total=len(df_merged)):
        if not row['txt']: continue
        counts = get_count_data_for_text(row['txt'])
        total = sum(counts.values())
        count = counts.get(word,0)
        fpm =  count / total * 1_000_000
        row['count'] = count
        row['total'] = total
        row['fpm'] = fpm
        ld.append(row)

    return pd.DataFrame(ld).sort_values('fpm', ascending=False)

In [ ]:
#@title Run it on a word
df = get_word_freq_historical_data('immigration')
df

In [ ]:
#@title TODO: Run it on another word
df2 = get_word_freq_historical_data('REPLACE_ME')
df2

In [ ]:
#@title (Function to plot historical frequencies)

def plot_word_freq_historical(word):
    df = get_word_freq_historical_data(word)
    df = df[df['Party'].isin(['Democrat','Republican'])]
    fig = p9.ggplot(df, p9.aes(x='year', y='fpm', color='Party'))
    fig += p9.geom_point()
    fig += p9.geom_smooth()
    return fig

In [ ]:
#@title Plot relative usage of "immigration" over time
plot_word_freq_historical('immigration')

In [ ]:
#@title Plot relative usage of another word over time
plot_word_freq_historical('REPLACE_ME')

## Key Word in Context (KWiC)

In [ ]:
# @title Use NLTK to produce concordance

@cache
def get_nltk_text(txt):
    tokens = nltk.word_tokenize(txt)
    text = nltk.Text(tokens)
    return text

@cache
def get_concordance(word):
    for i,row in tqdm(df_merged.iterrows(), total=len(df_merged)):
        if not row['txt']: continue
        txt_lowercase = row['txt'].lower()
        tokens = nltk.word_tokenize(txt_lowercase)
        if word not in tokens: continue
        text = get_nltk_text(row['txt'])
        print()
        print(row['id'])
        text.concordance(word, width=200)

In [ ]:
#@title Get instances of "immigration" in a speech
trump_2017_row = df_merged.query('id == "Trump_2017"').iloc[0]
trump_2017_txt = trump_2017_row['txt']
trump_2017_tokens = nltk.word_tokenize(trump_2017_txt)
trump_2017_nltk_text = nltk.Text(trump_2017_tokens)
trump_2017_nltk_text.concordance('immigration', width=200)

In [ ]:
#@title TODO: Get ALL instances of another word
get_concordance('REPLACE_ME')

## Statistical significance

In [ ]:
# @title Contingency table for word
def contingency_table_for_word(word):
    contingency_table = {
        'D': {
            'present': counts_D.get(word,0),
            'absent': total_D - counts_D.get(word,0),
        },
        'R': {
            'present': counts_R.get(word,0),
            'absent': total_R - counts_R.get(word,0)
        }
    }
    return pd.DataFrame(contingency_table)

contingency_table_for_word('immigration')

In [ ]:
# @title Compute statistical significance and obs/exp values

def contingency_stats_for_word(word):
    df_contingency = contingency_table_for_word(word)
    chi2, p, dof, expected = chi2_contingency(df_contingency)
    obs_exp_table = df_contingency / expected

    data = {
        'word': word,
        'chi2': chi2,
        'p': p,
        'is_signif': p < 0.05,
        'obs_exp_D': obs_exp_table.loc['present', 'D'],
        'obs_exp_R': obs_exp_table.loc['present', 'R'],
        'count_D': counts_D.get(word,0),
        'count_R': counts_R.get(word,0),
        'fpm_D': fpm_D.get(word,0),
        'fpm_R': fpm_R.get(word,0),
        'fpm_diff': fpm_R.get(word,0) - fpm_D.get(word,0)
    }
    return data

obs_exp_data = contingency_stats_for_word('immigration')
obs_exp_data

In [ ]:
# @title TODO: Find data for significantly different words

contingency_stats_for_word('REPLACE_ME')

In [ ]:
#@title Find data for all words
all_obs_exp_data = []
for word in tqdm(shared_words):
    obs_exp_data = contingency_stats_for_word(word)
    all_obs_exp_data.append(obs_exp_data)
df_all_obs_exp_data = pd.DataFrame(all_obs_exp_data)
df_all_obs_exp_data = df_all_obs_exp_data.sort_values('fpm_diff')
df_all_obs_exp_data

In [ ]:
#@title TODO: Click the calculator button on table above and 'narrativize' or interpret some distinctive words



In [ ]:
#@title TODO: Plot frequency of a word you're interested in from the above cell (scroll up to find the right function)



In [ ]:
#@title TODO: Get all occurrences of a word you're interested in from the above cell (scroll up to find the right function)

